In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PINN(nn.Module):
    def __init__(self, layers=[2, 64, 64, 64, 64, 1]):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers)-2):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
            self.layers.append(nn.Tanh())
        self.layers.append(nn.Linear(layers[-2], layers[-1]))
    
    def forward(self, X):
        for layer in self.layers:
            X = layer(X)
        return X

# Parameters
nu = 0.01 / np.pi
N_colloc = 20000
# Collocation points (interior)
colloc_x = 2 * torch.rand(N_colloc, 1, requires_grad=True, device=device) - 1  # x in [-1,1]
colloc_t = torch.rand(N_colloc, 1, requires_grad=True, device=device)          # t in [0,1]
colloc = torch.cat([colloc_x, colloc_t], dim=1)
colloc[:, 0] = 2 * colloc[:, 0] - 1  # x in [-1,1]
colloc[:, 1] = colloc[:, 1]          # t in [0,1]

# Initial condition points
ic_x = 2 * torch.rand(N_ic, 1, device=device) - 1
ic_t = torch.zeros_like(ic_x)
ic = torch.cat([ic_x, ic_t], dim=1)
ic.requires_grad = True
u_ic_true = -torch.sin(np.pi * ic_x)

model = PINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 15000
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # PDE residual
    u = model(colloc)
    grads = torch.autograd.grad(u, colloc, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_t = grads[:, 1:2]
    u_x = grads[:, 0:1]
    u_xx = torch.autograd.grad(u_x, colloc, grad_outputs=torch.ones_like(u_x), create_graph=True)[0][:, 0:1]
    
    residual = u_t + u * u_x - nu * u_xx
    loss_pde = torch.mean(residual**2)
    
    # Initial condition
    u_ic = model(ic)
    loss_ic = torch.mean((u_ic - u_ic_true)**2)
    
    # Periodic BCs approximated softly (or enforce exactly via network design—advanced)
    loss_bc = 0  # For simplicity here; in practice add points on boundaries
    
    loss = loss_pde + 10 * loss_ic  # Weight IC more if needed
    loss.backward(retain_graph=True)
    optimizer.step()
    
    if epoch % 2000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}, PDE: {loss_pde.item():.6f}, IC: {loss_ic.item():.6f}")

# Visualisation
with torch.no_grad():
    x = torch.linspace(-1, 1, 400)
    t = torch.linspace(0, 1, 200)
    X, T = torch.meshgrid(x, t, indexing='ij')
    XT = torch.stack([X.flatten(), T.flatten()], dim=1).to(device)
    u_pred = model(XT).cpu().numpy().reshape(400, 200)
    
    plt.figure(figsize=(10, 6))
    plt.contourf(T.numpy(), X.numpy(), u_pred.T, levels=50, cmap='viridis')
    plt.colorbar(label='u(x,t)')
    plt.xlabel('t')
    plt.ylabel('x')
    plt.title('PINN Solution to Burgers\' Equation')
    plt.show()

NameError: name 'N_ic' is not defined